<a href="https://colab.research.google.com/github/ingkapat/Thai-Scam-Call-Detector/blob/main/clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Install
!pip install pandas -q

In [2]:
# Cell 2: Upload dataset.jsonl จากคอม
from google.colab import files
uploaded = files.upload()
INPUT_FILENAME = list(uploaded.keys())[0]
print(f'Uploaded: {INPUT_FILENAME}')

Saving dataset.jsonl to dataset.jsonl
Uploaded: dataset.jsonl


In [3]:
# Cell 3: Import + Config
import json, re, os, random
import pandas as pd
from collections import defaultdict, Counter

INPUT_JSONL  = INPUT_FILENAME
OUT_DIR      = '/content/clean_output'
REVIEW_CSV   = os.path.join(OUT_DIR, 'dataset_review.csv')
EDITED_CSV   = os.path.join(OUT_DIR, 'dataset_review_edited.csv')
OUTPUT_JSONL = os.path.join(OUT_DIR, 'dataset_cleaned.jsonl')
SAMPLE_N     = 20

os.makedirs(OUT_DIR, exist_ok=True)

In [4]:
# Cell 4: Load dataset
data = []
with open(INPUT_JSONL, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        item = json.loads(line.strip())
        item['id'] = i
        data.append(item)

print(f'Total: {len(data)}')
print(f'Label 0: {sum(1 for d in data if d["label"]==0)}')
print(f'Label 1: {sum(1 for d in data if d["label"]==1)}')

Total: 22156
Label 0: 12015
Label 1: 10141


In [5]:
# Cell 5: สรุป category / subtype
print('Category:')
for k, v in Counter(d['category'] for d in data).most_common():
    print(f'  {k}: {v}')

print('\nSubtype:')
for k, v in Counter(d['subtype'] for d in data).most_common():
    print(f'  {k}: {v}')

Category:
  scam: 10141
  hard_negative: 2797
  official: 2025
  general_chat: 1962
  verification: 1487
  legit_promo: 1322
  delivery: 1224
  unknown: 1198

Subtype:
  phishing: 905
  loan_scam: 903
  sextortion: 894
  reward: 894
  otp_hijack: 890
  sms_alert: 888
  romance: 607
  impersonation: 604
  money_mule: 602
  tech_support: 596
  job_scam: 596
  fake_police_addline: 591
  investment: 589
  callcenter: 582
  gov_notice: 510
  company_hr: 508
  hospital: 505
  bank_real: 502
  reset_code: 499
  couple_talk: 497
  otp_english: 495
  otp_thai: 493
  family_check: 492
  colleague: 491
  friend_hangout: 482
  friend_new_number: 405
  friend_borrow: 404
  family_help: 402
  real_otp: 398
  colleague_expense: 398
  real_job: 397
  real_parcel: 393
  kerry_courier: 310
  ems_post: 308
  lazada_shopee: 306
  wrong_number: 306
  food_delivery: 300
  incomplete_speech: 299
  connection_issue: 297
  silent_call: 296
  insurance_offer: 270
  bank_offer: 267
  dtac_promo: 263
  true_promo

In [6]:
# Cell 6: Inspect samples per subtype (20 ตัว/subtype)
by_subtype = defaultdict(list)
for d in data:
    by_subtype[d['subtype']].append(d)

random.seed(42)
for subtype in sorted(by_subtype.keys()):
    items = by_subtype[subtype]
    samples = random.sample(items, min(SAMPLE_N, len(items)))
    print(f'\n=== {subtype} (label={samples[0]["label"]}, n={len(items)}) ===')
    for item in samples:
        print(f'\nid={item["id"]}')
        for t in item['turns']:
            print(f'  {t["speaker"]}: {t["text"]}')


=== ais_promo (label=0, n=259) ===

id=5030
  caller: ขออนุญาตครับ! สวัสดีค่ะ คุณลูกค้า ดิฉันจาก AIS ค่ะ
  user: อ่อ สวัสดีครับ
  caller: วันนี้เรามีโปรโมชันใหม่ค่ะ สำหรับแพ็กเกจ AIS Super Combo 699 บาท มีเน็ตไม่จำกัดนะคะ
  user: อืม แบบว่าเน็ตไม่จำกัดเลยเหรอครับ?
  caller: ใช่ค่ะ  มีโทรฟรีในเครือข่ายด้วยนะคะ คอนเฟิร์มไหมคะ?
  user: โอเคครับ คิดดูๆ แล้วนะ

id=1248
  caller: ขอโทษครับ คุณลูกค้า ผมจาก AIS ครับ
  user: อ๋อ ครับ
  caller: ตอนนี้เรามีโปรโมชั่นแพ็กเกจไม่ลดสปีด 499 บาท เพิ่มเน็ตให้ 20 GB ด้วยนะครับ
  user: แล้วมันดีมั้ยครับ?
  caller: ดีมากครับ เพราะใช้เน็ตเยอะ จะได้ไม่ต้องเสียค่าใช้จ่ายเพิ่มครับ
  user: โอเค เดี๋ยวลองคิดดูก่อนนะ

id=13065
  caller: เห้ สวัสดีครับ ผู้จัดการ AIS ครับ ไม่ทราบว่าคุณใช้งานแพ็กเกจอะไรอยู่ครับ?
  user: อ๋อ ใช้แพ็กเกจ 499 ครับ.
  caller: ดีมากเลยครับ ตอนนี้เรามีโปรโมชันใหม่ครับ เป็นแพ็กเกจ 699 บาท ได้เน็ต 20GB โทรฟรีในเครือข่าย ตลอด 24 ชั่วโมงนะครับ.
  user: อือ ฟังดูน่าสนใจนะ.
  caller: ใช่ครับ ถ้าสนใจสามารถเปลี่ยนได้เลยนะครับ จะให้ทำรายการเลยไหมค

In [7]:
# Cell 7: Remove unwanted ids / subtypes
remove_ids = []
remove_subtypes = []

data = [d for d in data
        if d['id'] not in set(remove_ids)
        and d['subtype'] not in set(remove_subtypes)]

print(f'Remaining: {len(data)}')

Remaining: 22156


In [8]:
# Cell 8: Clean text (ลบ ! ? ... emoji แต่เก็บลิงก์ไว้)
URL_PATTERN = re.compile(r'(https?://\S+|www\.\S+|bit\.ly/\S+)', re.IGNORECASE)
EMOJI_PATTERN = re.compile(
    '['
    '\U0001F600-\U0001F64F'
    '\U0001F300-\U0001F5FF'
    '\U0001F680-\U0001F6FF'
    '\U0001F1E0-\U0001F1FF'
    '\U00002500-\U00002BEF'
    '\U00002702-\U000027B0'
    '\U0001F900-\U0001F9FF'
    '\U0001FA00-\U0001FA6F'
    '\U0001FA70-\U0001FAFF'
    ']+', flags=re.UNICODE
)

def clean_text(text):
    urls = URL_PATTERN.findall(text)
    for i, url in enumerate(urls):
        text = text.replace(url, f'__URL{i}__', 1)
    text = EMOJI_PATTERN.sub('', text)
    text = re.sub(r'[!?]+', '', text)
    text = re.sub(r'\.{2,}', ' ', text)
    text = re.sub(r'[~^*<>{}\[\]\\|`]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    for i, url in enumerate(urls):
        text = text.replace(f'__URL{i}__', url)
    return text

for item in data:
    for turn in item['turns']:
        turn['text'] = clean_text(turn['text'])

print(f'Cleaned: {len(data)}')

Cleaned: 22156


In [9]:
# Cell 9: Export CSV เพื่อ review (จะ download มาที่เครื่อง)
rows = []
for d in data:
    full_text = ' | '.join(f'[{t["speaker"]}] {t["text"]}' for t in d['turns'])
    rows.append({
        'id': d['id'],
        'label': d['label'],
        'category': d['category'],
        'subtype': d['subtype'],
        'num_turns': len(d['turns']),
        'full_text': full_text,
        'remove': '',
    })

review_df = pd.DataFrame(rows)
review_df.to_csv(REVIEW_CSV, index=False, encoding='utf-8-sig')
print(f'Saved: {REVIEW_CSV}')

from google.colab import files
files.download(REVIEW_CSV)

Saved: /content/clean_output/dataset_review.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell 10: Upload CSV ที่แก้แล้วกลับมา (เปิดใน Excel ใส่ x ใน column remove)
from google.colab import files
uploaded = files.upload()
edited_filename = list(uploaded.keys())[0]

edited_df = pd.read_csv(edited_filename, encoding='utf-8-sig')
edited_df['remove'] = edited_df['remove'].fillna('').astype(str).str.strip().str.lower()
keep_df = edited_df[edited_df['remove'] != 'x']

print(f'Removed: {len(edited_df) - len(keep_df)}')
print(f'Remaining: {len(keep_df)}')

In [ ]:
# Cell 11: Export JSONL พร้อมไปเจนเสียง (download มาที่เครื่อง)
keep_ids = set(keep_df['id'].tolist())
cleaned_data = [d for d in data if d['id'] in keep_ids]

with open(OUTPUT_JSONL, 'w', encoding='utf-8') as f:
    for d in cleaned_data:
        out = {'label': d['label'], 'turns': d['turns']}
        f.write(json.dumps(out, ensure_ascii=False) + '\n')

print(f'Saved: {OUTPUT_JSONL}')
print(f'Total: {len(cleaned_data)}')

from google.colab import files
files.download(OUTPUT_JSONL)